# Classificacao de Imagens com CNN (Rede Neural Convolucional)

## Visao Geral

Este notebook demonstra como criar, treinar e fazer deploy de um modelo **CNN (Convolutional Neural Network)** para classificacao de imagens do dataset Fashion-MNIST.

### Por que usar CNN para imagens?

As CNNs sao especialmente eficazes para processamento de imagens porque:
- **Preservam relacoes espaciais** entre pixels
- **Detectam padroes locais** (bordas, texturas, formas)
- **Sao invariantes a translacao** (reconhecem objetos em qualquer posicao)
- **Requerem menos parametros** que redes totalmente conectadas

### Arquitetura da CNN

```
Input (28x28x1) 
    -> Conv2D (32 filtros) -> ReLU -> MaxPool
    -> Conv2D (64 filtros) -> ReLU -> MaxPool
    -> Flatten
    -> Dense (128) -> ReLU -> Dropout
    -> Dense (10) -> Softmax
```

In [ ]:
class FashionCNN(nn.Module):
    """
    Rede Neural Convolucional para classificacao de imagens Fashion-MNIST.
    
    Arquitetura:
    - 2 blocos convolucionais (Conv -> BatchNorm -> ReLU -> MaxPool)
    - 2 camadas fully connected com Dropout
    """
    
    def __init__(self, num_classes=10):
        super(FashionCNN, self).__init__()
        
        # Bloco Convolucional 1
        # Input: (1, 28, 28) -> Output: (32, 14, 14)
        self.conv1 = nn.Conv2d(
            in_channels=1,      # Imagem grayscale
            out_channels=32,    # 32 filtros
            kernel_size=3,      # Filtro 3x3
            padding=1           # Manter dimensao
        )
        self.bn1 = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)  # 28x28 -> 14x14
        
        # Bloco Convolucional 2
        # Input: (32, 14, 14) -> Output: (64, 7, 7)
        self.conv2 = nn.Conv2d(
            in_channels=32,
            out_channels=64,
            kernel_size=3,
            padding=1
        )
        self.bn2 = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)  # 14x14 -> 7x7
        
        # Bloco Convolucional 3 (opcional, para mais profundidade)
        # Input: (64, 7, 7) -> Output: (128, 3, 3)
        self.conv3 = nn.Conv2d(
            in_channels=64,
            out_channels=128,
            kernel_size=3,
            padding=1
        )
        self.bn3 = nn.BatchNorm2d(128)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)  # 7x7 -> 3x3
        
        # Camadas Fully Connected
        # Flatten: 128 * 3 * 3 = 1152
        self.fc1 = nn.Linear(128 * 3 * 3, 256)
        self.dropout1 = nn.Dropout(0.5)
        self.fc2 = nn.Linear(256, 128)
        self.dropout2 = nn.Dropout(0.3)
        self.fc3 = nn.Linear(128, num_classes)
        
    def forward(self, x):
        # Bloco 1
        x = self.conv1(x)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.pool1(x)
        
        # Bloco 2
        x = self.conv2(x)
        x = self.bn2(x)
        x = F.relu(x)
        x = self.pool2(x)
        
        # Bloco 3
        x = self.conv3(x)
        x = self.bn3(x)
        x = F.relu(x)
        x = self.pool3(x)
        
        # Flatten
        x = x.view(x.size(0), -1)
        
        # Fully Connected
        x = F.relu(self.fc1(x))
        x = self.dropout1(x)
        x = F.relu(self.fc2(x))
        x = self.dropout2(x)
        x = self.fc3(x)
        
        return x

# Criar modelo
model = FashionCNN(num_classes=10).to(device)

# Mostrar arquitetura
print("Arquitetura da CNN:")
print("=" * 60)
print(model)
print("=" * 60)

# Contar parametros
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal de parametros: {total_params:,}")
print(f"Parametros treinaveis: {trainable_params:,}")

## 3. Treinamento do Modelo

Vamos usar:
- **Otimizador**: Adam com learning rate 0.001
- **Loss**: CrossEntropyLoss (para classificacao multiclasse)
- **Scheduler**: ReduceLROnPlateau (reduz LR quando loss estagna)

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    """Treina uma epoca"""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    return running_loss / len(loader), correct / total


def evaluate(model, loader, criterion, device):
    """Avalia o modelo"""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    return running_loss / len(loader), correct / total


def train_model(model, train_loader, test_loader, epochs=10, lr=0.001):
    """Loop completo de treinamento"""
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=2, verbose=True
    )
    
    history = {
        'train_loss': [], 'train_acc': [],
        'val_loss': [], 'val_acc': []
    }
    
    best_acc = 0.0
    
    print("Iniciando treinamento...")
    print("=" * 70)
    
    for epoch in range(epochs):
        # Treinar
        train_loss, train_acc = train_epoch(
            model, train_loader, criterion, optimizer, device
        )
        
        # Avaliar
        val_loss, val_acc = evaluate(model, test_loader, criterion, device)
        
        # Scheduler
        scheduler.step(val_loss)
        
        # Salvar historico
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        
        # Salvar melhor modelo
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), 'best_cnn_model.pth')
        
        # Log
        current_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch+1:2d}/{epochs} | "
              f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | "
              f"LR: {current_lr:.6f}")
    
    print("=" * 70)
    print(f"Melhor acuracia de validacao: {best_acc:.4f}")
    
    return history

In [ ]:
# Treinar o modelo
EPOCHS = 10
history = train_model(model, train_loader, test_loader, epochs=EPOCHS, lr=0.001)

In [ ]:
# Visualizar historico de treinamento
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss
ax1.plot(history['train_loss'], label='Train Loss', marker='o')
ax1.plot(history['val_loss'], label='Val Loss', marker='s')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Loss durante Treinamento')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy
ax2.plot(history['train_acc'], label='Train Acc', marker='o')
ax2.plot(history['val_acc'], label='Val Acc', marker='s')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Acuracia durante Treinamento')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nAcuracia final de treino: {history['train_acc'][-1]:.4f}")
print(f"Acuracia final de validacao: {history['val_acc'][-1]:.4f}")

## 4. Avaliacao Detalhada

Vamos analisar a performance do modelo por classe usando matriz de confusao e metricas.

In [ ]:
# Carregar melhor modelo
model.load_state_dict(torch.load('best_cnn_model.pth', weights_only=True))
model.eval()

# Coletar todas as predicoes
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# Calcular acuracia por classe
print("Acuracia por Classe:")
print("=" * 40)
for i, name in enumerate(class_names):
    mask = all_labels == i
    acc = (all_preds[mask] == all_labels[mask]).mean()
    print(f"{name:15}: {acc:.4f}")

print("=" * 40)
print(f"Acuracia Geral: {(all_preds == all_labels).mean():.4f}")

In [ ]:
# Matriz de Confusao
from sklearn.metrics import confusion_matrix
import seaborn as sns

cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predito')
plt.ylabel('Real')
plt.title('Matriz de Confusao - CNN Fashion-MNIST')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Visualizar algumas predicoes
fig, axes = plt.subplots(3, 5, figsize=(15, 10))

# Pegar algumas imagens aleatorias
indices = np.random.choice(len(test_dataset), 15, replace=False)

for idx, ax in zip(indices, axes.flat):
    image, label = test_dataset[idx]
    
    # Predicao
    with torch.no_grad():
        output = model(image.unsqueeze(0).to(device))
        prob = F.softmax(output, dim=1)
        pred = output.argmax(dim=1).item()
        confidence = prob[0][pred].item()
    
    # Desnormalizar imagem
    img = image.squeeze().numpy() * 0.5 + 0.5
    
    ax.imshow(img, cmap='gray')
    color = 'green' if pred == label else 'red'
    ax.set_title(f"Pred: {class_names[pred]}\n({confidence:.1%})", 
                 color=color, fontsize=9)
    ax.set_xlabel(f"Real: {class_names[label]}", fontsize=8)
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle('Predicoes da CNN (Verde=Correto, Vermelho=Incorreto)', fontsize=12)
plt.tight_layout()
plt.show()

## 5. Visualizacao de Features

Vamos visualizar os filtros aprendidos pela CNN e os mapas de features gerados.

In [ ]:
# Visualizar filtros da primeira camada convolucional
filters = model.conv1.weight.data.cpu().numpy()

fig, axes = plt.subplots(4, 8, figsize=(14, 7))
for i, ax in enumerate(axes.flat):
    if i < filters.shape[0]:
        ax.imshow(filters[i, 0], cmap='gray')
        ax.set_title(f'F{i+1}', fontsize=8)
    ax.axis('off')

plt.suptitle('Filtros da Primeira Camada Convolucional (Conv1)', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Visualizar feature maps para uma imagem
def get_feature_maps(model, image):
    """Extrai feature maps de cada camada convolucional"""
    feature_maps = []
    x = image.unsqueeze(0).to(device)
    
    # Conv1
    x = model.conv1(x)
    x = model.bn1(x)
    x = F.relu(x)
    feature_maps.append(('Conv1', x.cpu().detach()))
    x = model.pool1(x)
    
    # Conv2
    x = model.conv2(x)
    x = model.bn2(x)
    x = F.relu(x)
    feature_maps.append(('Conv2', x.cpu().detach()))
    x = model.pool2(x)
    
    # Conv3
    x = model.conv3(x)
    x = model.bn3(x)
    x = F.relu(x)
    feature_maps.append(('Conv3', x.cpu().detach()))
    
    return feature_maps

# Pegar uma imagem de exemplo
sample_img, sample_label = test_dataset[0]
feature_maps = get_feature_maps(model, sample_img)

# Plotar feature maps
fig, axes = plt.subplots(3, 9, figsize=(16, 6))

# Imagem original
axes[0, 0].imshow(sample_img.squeeze().numpy() * 0.5 + 0.5, cmap='gray')
axes[0, 0].set_title(f'Original\n{class_names[sample_label]}', fontsize=8)
axes[0, 0].axis('off')

# Feature maps de cada camada
for row, (name, fmaps) in enumerate(feature_maps):
    fmaps = fmaps.squeeze().numpy()
    for col in range(min(8, fmaps.shape[0])):
        ax = axes[row, col + 1] if row == 0 else axes[row, col]
        ax.imshow(fmaps[col], cmap='viridis')
        if col == 0:
            ax.set_ylabel(name, fontsize=10)
        ax.axis('off')

# Limpar eixos nao usados
for ax in axes[1:, 8]:
    ax.axis('off')

plt.suptitle('Feature Maps da CNN', fontsize=12)
plt.tight_layout()
plt.show()

## 6. Salvar e Exportar o Modelo

Vamos salvar o modelo treinado em diferentes formatos para deploy.

In [ ]:
# Criar diretorio para artefatos
artifact_dir = Path('./model_artifacts_cnn')
artifact_dir.mkdir(exist_ok=True)

# 1. Salvar state_dict (recomendado para producao)
torch.save(model.state_dict(), artifact_dir / 'fashion_cnn.pth')

# 2. Salvar modelo completo
torch.save(model, artifact_dir / 'fashion_cnn_full.pth')

# 3. Exportar para TorchScript (para deploy otimizado)
model.eval()
example_input = torch.randn(1, 1, 28, 28).to(device)
traced_model = torch.jit.trace(model, example_input)
traced_model.save(str(artifact_dir / 'fashion_cnn_traced.pt'))

# 4. Salvar metadados
metadata = {
    'model_name': 'FashionCNN',
    'framework': 'PyTorch',
    'framework_version': torch.__version__,
    'input_shape': [1, 1, 28, 28],
    'output_classes': 10,
    'class_names': class_names,
    'accuracy': float(history['val_acc'][-1]),
    'created_at': datetime.now().isoformat(),
}

with open(artifact_dir / 'metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print("Artefatos salvos:")
for f in artifact_dir.iterdir():
    size = f.stat().st_size / 1024  # KB
    print(f"  {f.name}: {size:.1f} KB")

## 7. API REST para Deploy

Codigo para criar uma API Flask que serve o modelo CNN.

In [ ]:
# Criar API Flask para CNN
flask_code = '''"""API REST para modelo CNN Fashion-MNIST."""
from flask import Flask, request, jsonify
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import base64
from io import BytesIO
from PIL import Image

app = Flask(__name__)

class FashionCNN(nn.Module):
    def __init__(self, num_classes=10):
        super(FashionCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d(2, 2)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.pool3 = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(128 * 3 * 3, 256)
        self.dropout1 = nn.Dropout(0.5)
        self.fc2 = nn.Linear(256, 128)
        self.dropout2 = nn.Dropout(0.3)
        self.fc3 = nn.Linear(128, num_classes)
        
    def forward(self, x):
        x = self.pool1(F.relu(self.bn1(self.conv1(x))))
        x = self.pool2(F.relu(self.bn2(self.conv2(x))))
        x = self.pool3(F.relu(self.bn3(self.conv3(x))))
        x = x.view(x.size(0), -1)
        x = self.dropout1(F.relu(self.fc1(x)))
        x = self.dropout2(F.relu(self.fc2(x)))
        return self.fc3(x)

model = None
class_names = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

def load_model():
    global model
    model = FashionCNN()
    model.load_state_dict(torch.load("fashion_cnn.pth", weights_only=True))
    model.eval()
    return model

def preprocess_image(image_data):
    """Preprocessa imagem para o modelo"""
    if isinstance(image_data, str):  # Base64
        image_bytes = base64.b64decode(image_data)
        image = Image.open(BytesIO(image_bytes)).convert("L")
        image = image.resize((28, 28))
        image = np.array(image, dtype=np.float32)
    else:  # Array
        image = np.array(image_data, dtype=np.float32)
    
    # Normalizar
    if image.max() > 1:
        image = image / 255.0
    image = (image - 0.5) / 0.5
    
    # Adicionar dimensoes batch e channel
    if len(image.shape) == 2:
        image = image[np.newaxis, np.newaxis, :, :]
    
    return torch.tensor(image)

@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "healthy", "model": "FashionCNN"})

@app.route("/predict", methods=["POST"])
def predict():
    try:
        data = request.json
        image_data = data.get("image") or data.get("data")
        
        if image_data is None:
            return jsonify({"error": "Envie 'image' (base64) ou 'data' (array)"}), 400
        
        # Preprocessar
        tensor = preprocess_image(image_data)
        
        # Predicao
        with torch.no_grad():
            output = model(tensor)
            probs = F.softmax(output, dim=1)
            pred_class = output.argmax(dim=1).item()
            confidence = probs[0][pred_class].item()
        
        return jsonify({
            "class_id": pred_class,
            "class_name": class_names[pred_class],
            "confidence": confidence,
            "probabilities": {name: float(p) for name, p in zip(class_names, probs[0])}
        })
    except Exception as e:
        return jsonify({"error": str(e)}), 500

if __name__ == "__main__":
    load_model()
    print("CNN Model API rodando em http://localhost:5000")
    app.run(host="0.0.0.0", port=5000, debug=True)
'''

# Salvar API
with open(artifact_dir / "app.py", "w") as f:
    f.write(flask_code)

print(f"API salva em: {artifact_dir / 'app.py'}")
print(f"\nPara executar:")
print(f"  cd {artifact_dir}")
print(f"  python app.py")

## Resumo

Este notebook demonstrou como criar uma CNN para classificacao de imagens:

1. **Arquitetura**: 3 blocos convolucionais + 3 camadas fully connected
2. **Tecnicas**: BatchNorm, Dropout, Data Augmentation, Learning Rate Scheduler
3. **Resultados**: Acuracia superior a 90% no Fashion-MNIST
4. **Deploy**: API Flask pronta para producao

A CNN oferece melhor performance que redes totalmente conectadas (MLP) para tarefas de visao computacional.